In [1]:
import os
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [2]:
import pandas as pd

In [3]:
UEA_MTSC30 = ['EthanolConcentration',
              'FaceDetection',
              'Handwriting',
              'Heartbeat',
              'JapaneseVowels',
              'PEMS-SF',
              'SelfRegulationSCP1',
              'SelfRegulationSCP2',
              'SpokenArabicDigits',
              'UWaveGestureLibrary',
              'ArticularyWordRecognition',
              'AtrialFibrillation',
              'BasicMotions',
              'CharacterTrajectories',
              'Cricket',
              'DuckDuckGeese',
              'EigenWorms',
              'Epilepsy',
              'ERing',
              'FingerMovements',
              'HandMovementDirection',
              'InsectWingbeat',
              'Libras',
              'LSST',
              'MotorImagery',
              'NATOPS',
              'PenDigits',
              'PhonemeSpectra',
              'RacketSports',
              'StandWalkJump']

In [4]:
ckpt_dir = '/data/yoom618/TSLib/checkpoints/'
best_ckpt_dir = '/data/yoom618/TSLib/checkpoints_final/'
best_script_dir = '/data/yoom618/TSLib/scripts_final/'
full_results_dir = '/home/yoom618/TSLib/scripts_classification/03-full_results/MambaSL/additional (multilayer)'
model = 'MambaML'

for n_layers in [2,3]:
    print(f'==================={model}===================')
    os.makedirs(os.path.join(best_ckpt_dir, model), exist_ok=True)

    for data_name in UEA_MTSC30:

        if os.path.exists(f'{full_results_dir}/{model}_CLS_{data_name}_multilayer.out'):
            with open(f'{full_results_dir}/{model}_CLS_{data_name}_multilayer.out', 'r') as file:
                data = file.read().splitlines()
            with open(f'{full_results_dir}/{model}_CLS_{data_name}_multilayer.sh', 'r') as file:
                scripts = file.read().split('\n\n')[:-1]
                scripts = list(filter(lambda x: x[0] != '#', scripts))
        else:
            # print('no file')
            continue


        print(data_name)
        result_lst = list()
        for i in range(len(data)):
            if data[i].startswith('>>>>>>>testing : '):
                ckpt = data[i].replace('>>>>>>>testing : ', '').replace('<', '').strip()
                data_meta = ckpt.split('_')

                if data[i+2].startswith('gating_value'):
                    acc = float(data[i+3].replace('accuracy:', ''))
                    model_params = data[i+4].replace('model parameter : ', '')
                    model_size = data[i+5].replace('model size : ', '').replace('MB', '')
                else:
                    acc = float(data[i+2].replace('accuracy:', ''))
                    model_params = data[i+3].replace('model parameter : ', '')
                    model_size = data[i+4].replace('model size : ', '').replace('MB', '')

                result_data = {
                    i: data_meta[i] for i in range(9, len(data_meta))
                }
                result_data.update({
                    'acc': float(acc),
                    'model_params': int(model_params),
                    'model_size (MB)': float(model_size),
                    'ckpt': ckpt,
                    'n_layers': int(ckpt.split('_el')[1].split('_')[0])
                })

                result_lst.append(result_data)

        result_df = pd.DataFrame(reversed(result_lst))
        scripts = list(reversed(scripts))
        
        acc_max = result_df.loc[result_df['n_layers'] == n_layers]['acc'].max()
        result_acc_max = result_df.loc[(result_df['n_layers'] == n_layers) & (result_df['acc'] == acc_max)].sort_values(['model_size (MB)', 'model_params'])
        display(result_acc_max)



        # select best accuracy model (up to 20)
        result_acc_max = result_acc_max.iloc[:20]

        # move 20 lowest sized ckpts
        for ckpt in result_acc_max['ckpt']:
            if ckpt in os.listdir(ckpt_dir):
                os.makedirs(os.path.join(best_ckpt_dir, model), exist_ok=True)
                os.rename(os.path.join(ckpt_dir, ckpt), os.path.join(best_ckpt_dir, model, ckpt))
                print(f'> moved {ckpt}')
                

        assert len(result_df) == len(scripts), f'len(result_acc_max) != len(scripts) ({len(result_acc_max)} != {len(scripts)})'
        # make the scripts
        scripts_acc_max = f'''model_name="{model}"
dataset_name="{data_name}"
tslib_dir="/data/yoom618/TSLib"
gpu_id=0

data_dir="${{tslib_dir}}/dataset"
checkpoint_dir="${{tslib_dir}}/checkpoints_best/${{model_name}}"'''
        if len(result_acc_max) > 1:
            scripts_acc_max += '\n\n# below all have the same performance'

        for idx in result_acc_max.index:
            scripts_acc_max += '''\n\npython run.py \\
--use_gpu True \\
--gpu_type cuda \\
--gpu ${gpu_id} \\
--task_name classification \\
--data UEA \\
--root_path "${data_dir}/${dataset_name}" \\
--checkpoints ${checkpoint_dir} \\
--model ${model_name} \\
--model_id "CLS_${dataset_name}" \\
'''
            scripts_acc_max += '\n'.join(scripts[idx].replace("--is_training 1", "--is_training 0").split('\n')[15:])
        scripts_acc_max = scripts_acc_max.strip()
        out_dir = os.path.join(best_script_dir, model)
        script_fname = f'{out_dir}/{data_name}_nlayers{n_layers}.sh'
        if not os.path.exists(script_fname):
            os.makedirs(out_dir, exist_ok=True)
            with open(script_fname, 'w') as file:
                file.write(scripts_acc_max)
            print(f'> saved {data_name}_nlayers{n_layers}.sh')
            print()
    
        
        

===================MambaML===================
EthanolConcentration


,9,10,11,12,13,14,15,16,17,18,19,20,acc,model_params,model_size (MB),ckpt,n_layers
184,el2,dm512,ds8,expand1,dc4,nk36,tvdt0,tvB0,tvC0,useD0,gating4multilayer,0,0.410646,1699848,16.26,classification_CLS_EthanolConcentration_MambaM...,2


FaceDetection


,9,10,11,12,13,14,15,16,17,18,19,20,acc,model_params,model_size (MB),ckpt,n_layers
142,el2,dm256,ds4,expand1,dc4,nk3,tvdt1,tvB1,tvC0,useD0,gating4multilayer,0,0.687855,533000,6.93,classification_CLS_FaceDetection_MambaMultiLay...,2
234,el2,dm1024,ds16,expand1,dc4,nk3,tvdt0,tvB1,tvC0,useD0,gating4multilayer,0,0.687855,6989832,46.20,classification_CLS_FaceDetection_MambaMultiLay...,2


Handwriting


...

,9,10,11,12,13,14,15,16,17,18,19,20,acc,model_params,model_size (MB),ckpt,n_layers
391,el3,dm512,ds8,expand1,dc4,nk5,tvdt0,tvB1,tvC1,useD0,gating4multilayer,0,0.309872,2509832,19.35,classification_CLS_PhonemeSpectra_MambaMultiLa...,3


RacketSports


,9,10,11,12,13,14,15,16,17,18,19,20,acc,model_params,model_size (MB),ckpt,n_layers
317,el3,dm64,ds16,expand1,dc4,nk3,tvdt1,tvB0,tvC1,useD0,gating4multilayer,0,0.927632,51080,1.43,classification_CLS_RacketSports_MambaMultiLaye...,3
465,el3,dm1024,ds8,expand1,dc4,nk3,tvdt0,tvB0,tvC1,useD0,gating4multilayer,0,0.927632,9762824,56.79,classification_CLS_RacketSports_MambaMultiLaye...,3


StandWalkJump


,9,10,11,12,13,14,15,16,17,18,19,20,acc,model_params,model_size (MB),ckpt,n_layers
335,el3,dm128,ds2,expand1,dc4,nk50,tvdt1,tvB1,tvC1,useD0,gating4multilayer,0,0.666667,185992,3.16,classification_CLS_StandWalkJump_MambaMultiLay...,3
